#### Preparation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

In [2]:
DATA_DIR = r"C:\Users\yugil\Downloads\Cassava_Leaf_Datasets\Cassava_Leaf_Datasets\Classification"
WEIGHTS_DIR = "../weights"

BATCH_SIZE = 32
FREEZE_EPOCHS = 8
FINETUNE_EPOCHS = 20

LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
weights = ViT_B_16_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])


In [4]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)

Classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [5]:
# Load pre-trained ViT model
model = vit_b_16(weights=weights)

In [6]:
for param in model.parameters():
    param.requires_grad = False

In [7]:
in_features = model.heads.head.in_features

model.heads.head = nn.Linear(in_features, num_classes)

model = model.to(DEVICE)

In [8]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.heads.head.parameters(),
    lr=LR
)

In [9]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"\nTraining Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss, train_acc


In [10]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss, val_acc

#### Trainin 1 (Freeze BackBone)

In [11]:

print("\n-----------Starting Phase 1 Training-----------\n")

for epoch in range(FREEZE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FREEZE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FREEZE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")




-----------Starting Phase 1 Training-----------




Training Epoch 1/8: 100%|██████████| 30/30 [02:08<00:00,  4.28s/it]



Epoch [1/8]
Train Loss: 38.3179 | Train Acc: 0.4417
Val Loss: 9.5402 | Val Acc: 0.5708
Precision: 0.6171 | Recall: 0.5708 | F1: 0.5679



Training Epoch 2/8: 100%|██████████| 30/30 [02:05<00:00,  4.19s/it]



Epoch [2/8]
Train Loss: 33.2905 | Train Acc: 0.6510
Val Loss: 8.4470 | Val Acc: 0.6792
Precision: 0.7229 | Recall: 0.6792 | F1: 0.6776


...


Epoch [6/8]
Train Loss: 21.9977 | Train Acc: 0.7917
Val Loss: 6.1565 | Val Acc: 0.7667
Precision: 0.7853 | Recall: 0.7667 | F1: 0.7633



Training Epoch 7/8: 100%|██████████| 30/30 [02:04<00:00,  4.16s/it]



Epoch [7/8]
Train Loss: 20.5975 | Train Acc: 0.8094
Val Loss: 5.8447 | Val Acc: 0.7792
Precision: 0.7958 | Recall: 0.7792 | F1: 0.7764



Training Epoch 8/8: 100%|██████████| 30/30 [02:05<00:00,  4.19s/it]



Epoch [8/8]
Train Loss: 19.6410 | Train Acc: 0.8187
Val Loss: 5.5808 | Val Acc: 0.7792
Precision: 0.7963 | Recall: 0.7792 | F1: 0.7762


#### Training 2 (Fine-Tune)

In [12]:
for param in model.encoder.layers[-1].parameters():
    param.requires_grad = True

In [13]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

In [14]:
best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(FINETUNE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FINETUNE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FINETUNE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")


        
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, os.path.join(WEIGHTS_DIR, "Vit-B.pth"))


-----------Starting Fine-tuning Stage-----------




Training Epoch 1/20: 100%|██████████| 30/30 [02:04<00:00,  4.14s/it]



Epoch [1/20]
Train Loss: 17.6485 | Train Acc: 0.8260
Val Loss: 4.9984 | Val Acc: 0.7792
Precision: 0.7895 | Recall: 0.7792 | F1: 0.7769



Training Epoch 2/20: 100%|██████████| 30/30 [02:04<00:00,  4.13s/it]



Epoch [2/20]
Train Loss: 15.2658 | Train Acc: 0.8406
Val Loss: 4.4868 | Val Acc: 0.8083
Precision: 0.8161 | Recall: 0.8083 | F1: 0.8066


...


Epoch [18/20]
Train Loss: 4.4161 | Train Acc: 0.9563
Val Loss: 2.1187 | Val Acc: 0.9333
Precision: 0.9338 | Recall: 0.9333 | F1: 0.9332



Training Epoch 19/20: 100%|██████████| 30/30 [02:04<00:00,  4.15s/it]



Epoch [19/20]
Train Loss: 4.4288 | Train Acc: 0.9552
Val Loss: 2.1034 | Val Acc: 0.9250
Precision: 0.9252 | Recall: 0.9250 | F1: 0.9248



Training Epoch 20/20: 100%|██████████| 30/30 [02:03<00:00,  4.13s/it]



Epoch [20/20]
Train Loss: 4.1814 | Train Acc: 0.9604
Val Loss: 2.0454 | Val Acc: 0.9208
Precision: 0.9207 | Recall: 0.9208 | F1: 0.9204


#### Prediction Sample

In [15]:
checkpoint = torch.load("cassava_vitb16.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


C:\Users\yugil\AppData\Local\Temp\ipykernel_13648\77464708.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("cassava_vitb16.pth")


FileNotFoundError: [Errno 2] No such file or directory: 'cassava_vitb16.pth'